In [2]:
import torch
import torch.nn as nn
import random
import string


In [3]:
# CONFIG
D_MODEL = 128
N_HEADS = 4
N_LAYERS = 4
D_FF = 128
MAX_LEN = 20
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
# VOCAB
special_tokens = ["<PAD>", "<BOS>", "<SEP>", "<EOS>"]
letters = list(string.ascii_lowercase)

itos = special_tokens + letters
stoi = {ch: i for i, ch in enumerate(itos)}

PAD_ID = stoi["<PAD>"]
BOS_ID = stoi["<BOS>"]
SEP_ID = stoi["<SEP>"]
EOS_ID = stoi["<EOS>"]

VOCAB_SIZE = len(itos)

In [5]:
# MODEL
class MiniGPT(nn.Module):
    def __init__(self):
        super().__init__()

        self.token_emb = nn.Embedding(VOCAB_SIZE, D_MODEL)
        self.pos_emb = nn.Embedding(MAX_LEN, D_MODEL)

        decoder_layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL,
            nhead=N_HEADS,
            dim_feedforward=D_FF,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            decoder_layer,
            num_layers=N_LAYERS
        )

        self.lm_head = nn.Linear(D_MODEL, VOCAB_SIZE)

    def forward(self, input_ids):
        seq_len = input_ids.shape[1]

        pos = torch.arange(seq_len, device=input_ids.device).unsqueeze(0)
        x = self.token_emb(input_ids) + self.pos_emb(pos)

        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, device=input_ids.device),
            diagonal=1
        ).bool()

        x = self.transformer(x, mask=causal_mask)
        logits = self.lm_head(x)

        return logits


In [6]:
# DATA GENERATION
def generate_sample():
    length = random.randint(2, 6)
    x = [random.choice(letters) for _ in range(length)]
    y = list(reversed(x))
    return "".join(x), "".join(y)

In [7]:
# GENERATE FUNCTION
@torch.no_grad()
def generate(model, text, max_new_tokens=10):
    model.eval()

    tokens = ["<BOS>"] + list(text) + ["<SEP>"]
    ids = [stoi[t] for t in tokens]

    for _ in range(max_new_tokens):
        input_ids = torch.tensor(ids, dtype=torch.long, device=DEVICE).unsqueeze(0)

        if input_ids.shape[1] >= MAX_LEN:
            break

        logits = model(input_ids)
        next_id = torch.argmax(logits[0, -1]).item()

        ids.append(next_id)

        if next_id == EOS_ID:
            break

    decoded = [itos[i] for i in ids]
    return decoded

In [8]:
# EVALUATE
@torch.no_grad()
def evaluate_accuracy(model, n_samples=1000):
    exact_match = 0
    total_chars = 0
    correct_chars = 0

    for _ in range(n_samples):
        x, y = generate_sample()
        output_tokens = generate(model, x)

        if "<SEP>" in output_tokens:
            pred = output_tokens[output_tokens.index("<SEP>")+1:]
        else:
            pred = []

        if "<EOS>" in pred:
            pred = pred[:pred.index("<EOS>")]

        pred_str = "".join(pred)

        # Exact
        if pred_str == y:
            exact_match += 1

        # Char-level
        for p, t in zip(pred_str, y):
            if p == t:
                correct_chars += 1
            total_chars += 1

    return exact_match / n_samples, correct_chars / total_chars



In [9]:
# MAIN
if __name__ == "__main__":
    model = MiniGPT().to(DEVICE)

    model.load_state_dict(torch.load("minigpt_reverse_step1.pth", map_location=DEVICE))
    print("Loaded model")

    exact_acc, char_acc = evaluate_accuracy(model)

    print(f"Exact Accuracy: {exact_acc:.3f}")
    print(f"Char Accuracy: {char_acc:.3f}")

Loaded model
Exact Accuracy: 1.000
Char Accuracy: 1.000
